## Résultat Projet Data Engineering et au Machine Learning avec Snowflake

### Contexte

Les plateformes de données Snowflake propose une plateforme unifiée qui permet d’ingérer, transformer, analyser et entraîner des modèles de machine learning directement sur les données stockées dans la data plateforme, sans déplacer les données vers des systèmes externes.

Grâce à Snowpark, Snowflake ML et les Snowflake Notebooks, les data engineers et data scientists peuvent collaborer pour construire des pipelines de données et développer des modèles ML en utilisant des bibliothèques open source telles que scikit-learn, XGBoost ou PyTorch, tout en bénéficiant de l’élasticité et de la gouvernance de Snowflake.

Dans ce workshop, vous allez utiliser Snowflake comme plateforme complète de Data Engineering et Machine Learning en développant vos modèles ML directement dans l’environnement Snowflake.

### Consignes voir lien suivant https://github.com/atifrani/Modern-Data-Architecture/blob/main/Data-engineer-mlops/final_workshop.md

## Configuration de l’environnement

In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


## Etapes 1 : Ingestion et exploration des données

Database + Stage

In [ ]:
CREATE OR REPLACE DATABASE HOUSE_DB;
CREATE OR REPLACE STAGE house_price_stage
URL='s3://logbrain-datalake/datasets/house_price/'
FILE_FORMAT = (TYPE = JSON);

In [ ]:
LIST @house_price_stage;

Table RAW (JSON brut)

In [ ]:
CREATE OR REPLACE TABLE house_price_raw (
    data VARIANT);

Charger depuis S3

In [ ]:
COPY INTO house_price_raw
FROM @house_price_stage
FILE_FORMAT = (TYPE = JSON);

Etat de mon chargement 

In [ ]:
SELECT data
FROM house_price_raw
LIMIT 1;

In [ ]:
CREATE OR REPLACE TABLE house_price (
    price FLOAT,
    area FLOAT,
    bedrooms INT,
    bathrooms INT,
    stories INT,
    mainroad STRING,
    guestroom STRING,
    basement STRING,
    hotwaterheating STRING,
    airconditioning STRING,
    parking INT,
    prefarea STRING,
    furnishingstatus STRING
);

In [ ]:
INSERT INTO house_price
SELECT
    value:price::FLOAT,
    value:area::FLOAT,
    value:bedrooms::INT,
    value:bathrooms::INT,
    value:stories::INT,
    value:mainroad::STRING,
    value:guestroom::STRING,
    value:basement::STRING,
    value:hotwaterheating::STRING,
    value:airconditioning::STRING,
    value:parking::INT,
    value:prefarea::STRING,
    value:furnishingstatus::STRING
FROM house_price_raw,
LATERAL FLATTEN(input => data);

Chargement dans ma table 

In [ ]:
SELECT *
FROM house_price
LIMIT 10;

# Préparation des données

In [ ]:
df = session.table("HOUSE_PRICE").to_pandas()
df.head()


##  Etape 2 : Préparation des features + train/test split
Target = price

In [ ]:
# 2️⃣ Préparer les données pour ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Séparer features et cible
X = df.drop("PRICE", axis=1)
y = df["PRICE"]

# Convertir les colonnes catégorielles en variables numériques
X = pd.get_dummies(X, drop_first=True)

# Standardisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

## Etape 3 : Entraînement XGBoost + Linear Regression

### Xgboost

In [ ]:
# 3️⃣ Entraîner un modèle ML (XGBoost pour la régression)
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

# Création et entraînement du modèle
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, random_state=42)
xgb_model.fit(X_train, y_train)

# Prédictions
y_pred = xgb_model.predict(X_test)

# Évaluation
mse_xgboost = mean_squared_error(y_test, y_pred)
r2_xgboost = r2_score(y_test, y_pred)

print("XGBoost MSE:", mse_xgboost)
print("XGBoost R²:", r2_xgboost)

### Régression Linéaire 

In [ ]:
# 3️⃣ Entraîner un modèle de régression linéaire (équivalent "Logistic Regression" pour prix)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mse_linear_R = mean_squared_error(y_test, y_pred)
r2_linear_R = r2_score(y_test, y_pred)

print("Linear Regression MSE:", mse_linear_R)
print("Linear Regression R²:", r2_linear_R)

## Étape 4 : Comparaison et évaluation des modèles

### Comparaison Xgboost vs Linear_Regression

Évaluation basique (MSE, R²)

In [ ]:
# 5️⃣ Tableau comparatif
results = pd.DataFrame({
    "Model": ["Linear Regression", "XGBoost"],
    "MSE": [mse_linear_R, mse_xgboost],
    "R²": [r2_linear_R, r2_xgboost]
})

print(results)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Récapitulatif des performances
results = {
    "Linear Regression": {"MSE": mse_linear_R, "R²": r2_linear_R},
    "XGBoost":           {"MSE": mse_xgboost,  "R²": r2_xgboost},
}

print("=" * 45)
print(f"{'Modèle':<22} {'MSE':>10} {'R²':>10}")
print("=" * 45)
for name, m in results.items():
    print(f"{name:<22} {m['MSE']:>10.0f} {m['R²']:>10.4f}")
print("=" * 45)

# Visualisation : valeurs réelles vs prédites (XGBoost)
y_pred_xgb = xgb_model.predict(X_test)
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_xgb, alpha=0.5, color="#1D9E75")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Prix réel")
plt.ylabel("Prix prédit")
plt.title("XGBoost — Réel vs Prédit")
plt.tight_layout()
plt.show()

### Analyse des métriques globales

In [ ]:
# 1️⃣ Transformer PRICE en classes
df['PRICE_CLASS'] = pd.qcut(df['PRICE'], q=3, labels=['bas', 'moyen', 'élevé'])

# Encoder UNE FOIS
df['PRICE_CLASS_ENC'] = df['PRICE_CLASS'].astype('category').cat.codes

# Features et target
X = df.drop(columns=['PRICE', 'PRICE_CLASS', 'PRICE_CLASS_ENC'])
y = df['PRICE_CLASS_ENC']

# Dummies
X = pd.get_dummies(X, drop_first=True)

# Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# 2️⃣ Modèle
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 3️⃣ Prédictions
y_pred = model.predict(X_test)

# 4️⃣ Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))

print("\nDétails complets :")
print(classification_report(y_test, y_pred))

## Analyse des métriques globales
Accuracy = 0.734 → 73,4 % des prédictions sont correctes.
Precision (weighted) = 0.733 → quand le modèle prédit une classe, il a raison environ 73 % du temps.
Recall (weighted) = 0.734 → le modèle capture environ 73 % de toutes les instances réelles.

Ces valeurs montrent que ton modèle fonctionne correctement, mais il y a des différences entre les classes.

### Analyse par classe
Classe	Precision	Recall	F1-score	Support	Interprétation
bas	0.72	0.73	0.72	66	Bon équilibre, le modèle prédit et capture bien.
moyen	0.74	0.60	0.66	84	Le recall est plus faible → beaucoup d’instances moyennes sont mal classées.
élevé	0.75	0.91	0.82	68	Très bon recall → la plupart des prix élevés sont correctement détectés.

Observation clé :

La classe moyen est la plus problématique : beaucoup de prix moyens sont confondus avec bas ou élevé.

La classe élevé est très bien détectée.

### Axes d’amélioration
Rééquilibrage des classes
Utiliser class_weight='balanced' dans Logistic Regression ou XGBoost Classifier.
Ou générer plus d’exemples pour la classe moyenne si elle est sous-représentée.
Feature Engineering
Ajouter des variables ou interactions qui aident à mieux distinguer les prix moyens.
Exemple : surface / nb_pieces, année_construction, quartier.
Hyperparameter Tuning
Pour XGBoost ou Logistic Regression, tester plusieurs paramètres (max_depth, learning_rate, C, penalty) pour améliorer le recall sur la classe moyenne.
Modèles plus robustes
Essayer RandomForest, Gradient Boosting, LightGBM pour mieux gérer les distributions complexes et les classes déséquilibrées.
Analyse des erreurs
Examiner les faux négatifs et faux positifs pour la classe « moyen ».
Identifier les patterns qui provoquent les confusions et créer des features spécifiques.

### Résumé :

Modèle global correct (~73 % accuracy)
Classe moyenne sous-performante → priorité d’amélioration
Solutions : rééquilibrage, features supplémentaires, tuning hyperparamètres, modèles plus complexes.

## Étape 5 : Optimisation avec hyperparamètres (RandomizedSearchCV)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# S'assurer que X_train est bien un array numpy propre
X_train_arr = np.array(X_train)
X_test_arr  = np.array(X_test)

param_dist = {
    "n_estimators":     [50, 100, 200, 300],
    "max_depth":        [3, 5, 7, 9],
    "learning_rate":    [0.01, 0.05, 0.1, 0.2],
    "subsample":        [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
}

xgb_base = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    enable_categorical=False
)

random_search = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_dist,
    n_iter=20,          # réduit pour éviter timeout
    scoring="r2",
    cv=3,               # réduit aussi
    verbose=1,
    random_state=42,
    n_jobs=1            # -1 peut poser problème dans Snowflake
)

random_search.fit(X_train_arr, y_train)

best_model = random_search.best_estimator_
print("Meilleurs hyperparamètres :", random_search.best_params_)

y_pred_best = best_model.predict(X_test_arr)
mse_best = mean_squared_error(y_test, y_pred_best)
r2_best  = r2_score(y_test, y_pred_best)

print(f"Meilleur modèle — MSE : {mse_best:.0f} | R² : {r2_best:.4f}")

## Étape 6 : Stockage dans le Snowflake Model Registry

In [ ]:
from snowflake.ml.registry import Registry

# Initialiser le registry
reg = Registry(session=session, database_name="HOUSE_DB", schema_name="PUBLIC")

# Sauvegarder le meilleur modèle
reg.log_model(
    model=best_model,
    model_name="HOUSE_PRICE_PREDICTOR",
    version_name="v1",
    comment="XGBoost optimisé — RandomizedSearchCV — R²: " + str(round(r2_best, 4)),
    metrics={"MSE": mse_best, "R2": r2_best},
    sample_input_data=X_test[:5]
)

print("Modèle enregistré dans le Snowflake Model Registry.")

## Étape 7 : Inférence depuis le Model Registry

In [ ]:
import pandas as pd
import numpy as np

# Récupérer les noms de colonnes exacts attendus par le modèle
expected_columns = [f"input_feature_{i}" for i in range(X_test.shape[1])]

# Recréer X_test avec les bons noms de colonnes
X_test_named = pd.DataFrame(X_test, columns=expected_columns)

# Charger le modèle depuis le registry
loaded_model = reg.get_model("HOUSE_PRICE_PREDICTOR").version("v1")

# Générer les prédictions sur les 10 premières lignes
new_data = X_test_named.iloc[:10]
predictions = loaded_model.run(new_data, function_name="predict")

print("Prédictions :")
print(predictions)

# Stocker les résultats dans une table Snowflake
results_df = pd.DataFrame({
    "PREDICTED_PRICE": predictions.values.flatten(),
    "ACTUAL_PRICE":    y_test[:10].values
})

snow_df = session.create_dataframe(results_df)
snow_df.write.mode("overwrite").save_as_table("HOUSE_DB.PUBLIC.HOUSE_PRICE_PREDICTIONS")
print("Prédictions stockées dans HOUSE_PRICE_PREDICTIONS.")

## Étape 8 : Application Streamlit

In [ ]:
# Coller ce code dans un fichier Streamlit in Snowflake

import streamlit as st
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
import pandas as pd

session = get_active_session()

st.title("Estimation du prix d'une maison")
st.markdown("Renseignez les caractéristiques du bien pour obtenir une estimation.")

# Inputs utilisateur
col1, col2 = st.columns(2)
with col1:
    area       = st.number_input("Surface (m²)",         min_value=20,  max_value=1000, value=100)
    bedrooms   = st.slider("Chambres",                   1, 10, 3)
    bathrooms  = st.slider("Salles de bain",             1, 5,  1)
    stories    = st.slider("Étages",                     1, 4,  1)
    parking    = st.slider("Places de parking",          0, 5,  1)
with col2:
    mainroad        = st.selectbox("Route principale",   ["yes", "no"])
    guestroom       = st.selectbox("Chambre d'amis",     ["yes", "no"])
    basement        = st.selectbox("Sous-sol",           ["yes", "no"])
    hotwaterheating = st.selectbox("Chauffage eau chaude",["yes", "no"])
    airconditioning = st.selectbox("Climatisation",      ["yes", "no"])
    prefarea        = st.selectbox("Zone privilégiée",   ["yes", "no"])
    furnishingstatus= st.selectbox("Ameublement",        ["furnished", "semi-furnished", "unfurnished"])

if st.button("Estimer le prix"):
    input_df = pd.DataFrame([{
        "area": area, "bedrooms": bedrooms, "bathrooms": bathrooms,
        "stories": stories, "parking": parking,
        "mainroad": mainroad, "guestroom": guestroom,
        "basement": basement, "hotwaterheating": hotwaterheating,
        "airconditioning": airconditioning, "prefarea": prefarea,
        "furnishingstatus": furnishingstatus
    }])

    # Encodage identique à l'entraînement
    input_encoded = pd.get_dummies(input_df, drop_first=True)
    input_encoded = input_encoded.reindex(columns=feature_columns, fill_value=0)

    reg = Registry(session=session, database_name="HOUSE_DB", schema_name="PUBLIC")
    model = reg.get_model("HOUSE_PRICE_PREDICTOR").version("v1")

    prediction = model.run(input_encoded, function_name="predict")
    price = prediction.values[0][0]

    st.success(f"Prix estimé : {price:,.0f} €")